In [ ]:
# %% [markdown]
# # Perturbation PSF — cell-type-resolved spatial response kernels
#
# Estimates empirical **response kernels K(r)**: the excess molecular-program
# activity in barcode-negative recipient cells as a function of distance *r* from
# sparse barcode-positive perturbed source cells, versus a matched safe-harbour
# (msafe) control, on mouse-brain Spatial Perturb-seq (Stereo-seq, GSE274447).
#
# Pipeline: raw .gef → unified per-cell table → cell typing + niches →
# source-centered neighborhoods → pathway program scores → spline kernel K(r)
# with source-clustered inference → null calibration → synthetic recovery.
#
# Run end-to-end for all 18 perturbations via `psf_pipeline.run_all()`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import psf_pipeline as psf

## 1. Load typed cells + program scores (built by Steps 1–5)

In [ ]:
tc, ps = psf.load("checkpoints")
print(f"{len(tc):,} cells | {tc.cell_type.nunique()} types | {tc.niche.nunique()} niches")
print(tc.perturbation.value_counts().head(20))

## 2. Single-guide source counts (guide UMI threshold ≥2)

In [ ]:
counts = tc.perturbation.value_counts()
guides = [p for p in counts.index if p not in ("Neg", "Multiple")]
print(counts[guides])

## 3. Fit one kernel: Lrrk2 × LRP1 signaling (with 95% CI)

In [ ]:
allsrc = tc[tc.perturbation.isin(["lrrk2", "msafe"])]
nb = psf.build_neighborhoods(tc, allsrc, rmax=250)
progs = list(pd.read_csv("checkpoints/programs_used.csv").program)
af = psf.make_frame(nb, tc, ps, progs)
res = psf.fit_kernel(af, "lrrk2", "LRP1_signaling")
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(res.r, res.K, color="#d62728", lw=2)
ax.fill_between(res.r, res.lo, res.hi, color="#d62728", alpha=0.15)
ax.axhline(0, color="k", lw=0.6, ls="--")
ax.set_xlabel("distance r (µm)"); ax.set_ylabel("excess K(r)")
ax.set_title("Lrrk2 → LRP1 signaling kernel")
fig.tight_layout()

## 4. Run all perturbations end-to-end

In [ ]:
Kdf, sdf = psf.run_all("checkpoints", "checkpoints", rmax=250, min_sources=5)
print(f"{sdf.perturbation.nunique()} perturbations × {sdf.program.nunique()} programs "
      f"= {len(sdf)} kernels; {int(sdf.peak_sig.sum())} significant peaks")

## 5. Key result
Under source-clustered inference with niche/section/cell-type/depth adjustment,
the paper's neighbour DEGs (Lrrk2→Sparc↓, Lrp1↓, Vps35l↑) reproduce in
*direction* at the single-gene level but do **not** reach significance. Synthetic
recovery shows the 2σ detection floor (~0.34 near-source gain) sits above the
real effect sizes at the available source counts (Lrrk2 n=28, Cfap410 n=51
single-guide sources; msafe control n=50, ~25 per arm in the synthetic split)
— a power limit, quantified, not ignored.